In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Image Authenticity Module — Hive Moderation API

Uses the Hive AI-Generated Image Detection API to score whether an image is
real (camera-captured) or AI-generated. Returns a score between 0 (real) and 1 (AI-generated).

In [1]:
import json
import os
import requests
import time

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# ════════════════════════════════════════════════════════
# PASTE YOUR HIVE API KEY HERE
# Sign up at: https://thehive.ai  →  Get API Key
# ════════════════════════════════════════════════════════
HIVE_API_KEY = "OPDOhFbLSUItcAwqeOzXKQ=="

assert HIVE_API_KEY != "YOUR_API_KEY_HERE", "Please set your Hive API key above!"
print(f"API key set: {HIVE_API_KEY[:8]}...{HIVE_API_KEY[-4:]}")

API key set: OPDOhFbL...KQ==


In [2]:
# ── Config & data loading ──
DATASET_ROOT     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")
IMAGE_BASE       = os.path.join(DATASET_ROOT, "origin")

def resolve_image_path(meta_image_path: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGE_BASE, rel)

with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Grab 5 unique images
seen_images = set()
samples = []
for ann in annotations:
    if len(samples) >= 5:
        break
    img_id = str(ann["image_id"])
    if img_id in seen_images or img_id not in metadata:
        continue
    img_path = resolve_image_path(metadata[img_id]["image_path"])
    if not os.path.isfile(img_path):
        continue
    seen_images.add(img_id)
    samples.append({
        "image_id": int(img_id),
        "image_path": img_path,
        "caption": metadata[img_id]["caption"],
        "source": metadata[img_id]["source"],
    })

print(f"Selected {len(samples)} images:")
for s in samples:
    print(f"  [{s['source']}] {s['caption'][:80]}...")

Selected 5 images:
  [guardian] Saudi troops cheer as they ride at the back of an army truck in the southwestern...
  [guardian] Israeli soldiers ride on a tank to a position near the Israel Gaza border...
  [bbc] Tiffany is north east of Aberdeen...
  [bbc] Some ferry services were affected between Holyhead and Dublin on Friday...
  [bbc] Rock band Paramore is another group which sells cruises...


In [12]:
# ── Hive AI-Generated Image Detection function (v3 API) ──

import base64

HIVE_API_URL = "https://api.thehive.ai/api/v3/hive/ai-generated-and-deepfake-content-detection"

def hive_ai_detection(image_path: str, api_key: str = HIVE_API_KEY) -> dict:
    """
    Send an image to Hive v3 AI-generated content detection endpoint.
    Returns ai_generated_score (0-1) and not_ai_generated_score (0-1).
    """
    ext = os.path.splitext(image_path)[1].lower()
    mime = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png",
            ".bmp": "image/bmp", ".webp": "image/webp"}.get(ext, "image/jpeg")

    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    payload = {
        "media_metadata": True,
        "input": [{"media_base64": f"data:{mime};base64,{b64}"}],
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    response = requests.post(HIVE_API_URL, headers=headers, data=json.dumps(payload), timeout=60)
    if response.status_code != 200:
        print(f"  Response {response.status_code}: {response.text[:300]}")
    response.raise_for_status()
    resp_data = response.json()

    # Parse v3 response — uses "value" not "score"
    ai_score = None
    not_ai_score = None
    raw_classes = []

    for output in resp_data.get("output", []):
        for cls in output.get("classes", []):
            raw_classes.append({"class": cls["class"], "value": cls["value"]})
            if cls["class"] == "ai_generated":
                ai_score = cls["value"]
            elif cls["class"] == "not_ai_generated":
                not_ai_score = cls["value"]

    return {
        "ai_generated_score": ai_score,
        "not_ai_generated_score": not_ai_score,
        "raw_classes": raw_classes,
    }

# Quick test
print("Testing API connection (v3)...")
test_result = hive_ai_detection(samples[0]["image_path"])
print(f"Success! AI score: {test_result['ai_generated_score']:.6f} | Real score: {test_result['not_ai_generated_score']:.6f}")

Testing API connection (v3)...
Success! AI score: 0.000317 | Real score: 0.999683


In [13]:
# ── Test on 5 images ──
results = []

for i, s in enumerate(samples):
    print(f"[{i+1}/{len(samples)}] Sending {os.path.basename(s['image_path'])}...")

    try:
        hive_result = hive_ai_detection(s["image_path"])
        ai_score = hive_result["ai_generated_score"]
        real_score = hive_result["not_ai_generated_score"]

        results.append({
            "image_id": s["image_id"],
            "source": s["source"],
            "caption": s["caption"][:60] + "...",
            "ai_generated_score": ai_score,
            "not_ai_generated_score": real_score,
            "verdict": "AI-GENERATED" if ai_score and ai_score > 0.5 else "REAL",
        })

        print(f"  AI: {ai_score:.4f} | Real: {real_score:.4f} | {'AI-GENERATED' if ai_score > 0.5 else 'REAL'}")
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "image_id": s["image_id"],
            "source": s["source"],
            "caption": s["caption"][:60] + "...",
            "ai_generated_score": None,
            "not_ai_generated_score": None,
            "verdict": "ERROR",
        })

    time.sleep(0.5)

print("\nDone!")

[1/5] Sending 482.jpg...
  AI: 0.0003 | Real: 0.9997 | REAL
[2/5] Sending 113.jpg...
  AI: 0.0001 | Real: 0.9999 | REAL
[3/5] Sending 584.jpg...
  AI: 0.0000 | Real: 1.0000 | REAL
[4/5] Sending 036.jpg...
  AI: 0.0004 | Real: 0.9996 | REAL
[5/5] Sending 675.jpg...
  AI: 0.0004 | Real: 0.9996 | REAL

Done!


In [14]:
# ── Results table ──
df = pd.DataFrame(results)
display(df[["image_id", "source", "ai_generated_score", "not_ai_generated_score", "verdict", "caption"]])

valid = df[df["ai_generated_score"].notna()]
if len(valid) > 0:
    print(f"\nAvg AI-generated score: {valid['ai_generated_score'].mean():.4f}")
    print(f"Avg Real score:         {valid['not_ai_generated_score'].mean():.4f}")
    print(f"All classified as REAL: {(valid['verdict'] == 'REAL').all()}")
    print("\n(These are real news photos, so we expect low AI scores)")

,image_id,source,ai_generated_score,not_ai_generated_score,verdict,caption
0,620701,guardian,0.000317,0.999683,REAL,Saudi troops cheer as they ride at the back of...
1,1179991,guardian,0.000111,0.999889,REAL,Israeli soldiers ride on a tank to a position ...
2,1569820,bbc,0.000015,0.999985,REAL,Tiffany is north east of Aberdeen...
3,1583815,bbc,0.000440,0.999560,REAL,Some ferry services were affected between Holy...
4,1490080,bbc,0.000377,0.999623,REAL,Rock band Paramore is another group which sell...



Avg AI-generated score: 0.0003
Avg Real score:         0.9997
All classified as REAL: True

(These are real news photos, so we expect low AI scores)


In [16]:
# ── Side-by-side: Real News vs AI-Generated ──
TEST_DIR = _os.path.join(str(_cfg.ROOT), 'test_images')

real_images = sorted([
    {"path": os.path.join(TEST_DIR, "real_news", f), "label": "REAL", "name": f}
    for f in os.listdir(os.path.join(TEST_DIR, "real_news")) if f.endswith((".jpg", ".png"))
], key=lambda x: x["name"])

ai_images = sorted([
    {"path": os.path.join(TEST_DIR, "ai_generated", f), "label": "AI-GEN", "name": f}
    for f in os.listdir(os.path.join(TEST_DIR, "ai_generated")) if f.endswith((".jpg", ".png"))
], key=lambda x: x["name"])

all_test = real_images + ai_images
print(f"Real news photos: {len(real_images)}")
print(f"AI-generated images: {len(ai_images)}")
print(f"Total: {len(all_test)}")
for img in all_test:
    print(f"  [{img['label']}] {img['name']}")

Real news photos: 4
AI-generated images: 4
Total: 8
  [REAL] real_news_1.jpg
  [REAL] real_news_2.jpg
  [REAL] real_news_3.jpg
  [REAL] real_news_4.jpg
  [AI-GEN] ai_face_1.jpg
  [AI-GEN] ai_face_2.jpg
  [AI-GEN] ai_face_3.jpg
  [AI-GEN] ai_face_4.jpg


In [17]:
# ── Run Hive API on all test images ──
comparison_results = []

for i, img in enumerate(all_test):
    print(f"[{i+1}/{len(all_test)}] {img['label']:6s} | {img['name']}...", end=" ")

    try:
        result = hive_ai_detection(img["path"])
        ai_score = result["ai_generated_score"]
        real_score = result["not_ai_generated_score"]
        verdict = "AI-GENERATED" if ai_score and ai_score > 0.5 else "REAL"
        correct = (img["label"] == "REAL" and verdict == "REAL") or (img["label"] == "AI-GEN" and verdict == "AI-GENERATED")

        comparison_results.append({
            "filename": img["name"],
            "ground_truth": img["label"],
            "ai_score": ai_score,
            "real_score": real_score,
            "verdict": verdict,
            "correct": correct,
            "path": img["path"],
        })
        mark = "Y" if correct else "X"
        print(f"AI={ai_score:.4f} | {verdict} [{mark}]")
    except Exception as e:
        print(f"ERROR: {e}")
        comparison_results.append({
            "filename": img["name"],
            "ground_truth": img["label"],
            "ai_score": None,
            "real_score": None,
            "verdict": "ERROR",
            "correct": False,
            "path": img["path"],
        })

    time.sleep(0.5)

print("\nDone!")

[1/8] REAL   | real_news_1.jpg... AI=0.0000 | REAL [Y]
[2/8] REAL   | real_news_2.jpg... AI=0.0000 | REAL [Y]
[3/8] REAL   | real_news_3.jpg... AI=0.0014 | REAL [Y]
[4/8] REAL   | real_news_4.jpg... AI=0.0000 | REAL [Y]
[5/8] AI-GEN | ai_face_1.jpg... AI=1.0000 | AI-GENERATED [Y]
[6/8] AI-GEN | ai_face_2.jpg... AI=1.0000 | AI-GENERATED [Y]
[7/8] AI-GEN | ai_face_3.jpg... AI=1.0000 | AI-GENERATED [Y]
[8/8] AI-GEN | ai_face_4.jpg... AI=0.9999 | AI-GENERATED [Y]

Done!


In [18]:
# ── Results table & accuracy ──
cdf = pd.DataFrame(comparison_results)
display(cdf[["filename", "ground_truth", "ai_score", "real_score", "verdict", "correct"]])

valid = cdf[cdf["ai_score"].notna()]
acc = valid["correct"].mean()

real_rows = valid[valid["ground_truth"] == "REAL"]
ai_rows = valid[valid["ground_truth"] == "AI-GEN"]

print(f"\n{'=' * 60}")
print(f"HIVE AI-GENERATED DETECTION — COMPARISON RESULTS")
print(f"{'=' * 60}")
print(f"  Accuracy: {acc:.0%} ({valid['correct'].sum()}/{len(valid)})")
print(f"\n  REAL news photos  — avg AI score: {real_rows['ai_score'].mean():.6f}")
print(f"  AI-generated      — avg AI score: {ai_rows['ai_score'].mean():.6f}")
print(f"  Separation        : {ai_rows['ai_score'].mean() - real_rows['ai_score'].mean():.6f}")

,filename,ground_truth,ai_score,real_score,verdict,correct
0,real_news_1.jpg,REAL,0.000024,0.999976,REAL,True
1,real_news_2.jpg,REAL,0.000015,0.999985,REAL,True
2,real_news_3.jpg,REAL,0.001433,0.998567,REAL,True
3,real_news_4.jpg,REAL,0.000007,0.999993,REAL,True
4,ai_face_1.jpg,AI-GEN,0.999981,0.000019,AI-GENERATED,True
5,ai_face_2.jpg,AI-GEN,0.999991,0.000009,AI-GENERATED,True
6,ai_face_3.jpg,AI-GEN,0.999983,0.000017,AI-GENERATED,True
7,ai_face_4.jpg,AI-GEN,0.999927,0.000073,AI-GENERATED,True



HIVE AI-GENERATED DETECTION — COMPARISON RESULTS
  Accuracy: 100% (8/8)

  REAL news photos  — avg AI score: 0.000370
  AI-generated      — avg AI score: 0.999970
  Separation        : 0.999601


In [ ]:
# ── Visual comparison grid ──
n = len(comparison_results)
fig, axes = plt.subplots(2, n // 2, figsize=(5 * (n // 2), 10))

# Top row: real, bottom row: AI-generated
real_items = [r for r in comparison_results if r["ground_truth"] == "REAL"]
ai_items = [r for r in comparison_results if r["ground_truth"] == "AI-GEN"]

for row_idx, (items, row_label) in enumerate([(real_items, "Real News Photos"), (ai_items, "AI-Generated (StyleGAN)")]):
    for col_idx, item in enumerate(items):
        ax = axes[row_idx, col_idx]
        img = Image.open(item["path"]).convert("RGB")
        ax.imshow(img)

        ai_s = item["ai_score"]
        color = "green" if item["correct"] else "red"
        score_str = f"{ai_s:.4f}" if ai_s is not None else "ERR"

        ax.set_title(
            f"GT: {item['ground_truth']} | Pred: {item['verdict']}\nAI score: {score_str}",
            fontsize=10, color=color, fontweight="bold"
        )
        ax.set_xlabel(item["filename"], fontsize=8)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    # If fewer items than columns, hide extra axes
    for col_idx in range(len(items), n // 2):
        axes[row_idx, col_idx].axis("off")

fig.text(0.01, 0.75, "REAL", fontsize=14, fontweight="bold", rotation=90, va="center", color="#2ecc71")
fig.text(0.01, 0.25, "AI-GEN", fontsize=14, fontweight="bold", rotation=90, va="center", color="#e74c3c")

plt.suptitle("Hive API: Real News Photos vs AI-Generated Images", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0.02, 0, 1, 0.96])
plt.savefig("hive_real_vs_ai_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Bar chart comparison ──
fig, ax = plt.subplots(figsize=(10, 5))

labels = [r["filename"] for r in comparison_results]
scores = [r["ai_score"] or 0 for r in comparison_results]
colors = ["#2ecc71" if r["ground_truth"] == "REAL" else "#e74c3c" for r in comparison_results]

bars = ax.bar(range(len(labels)), scores, color=colors, edgecolor="white", alpha=0.85)
ax.axhline(y=0.5, color="orange", linestyle="--", linewidth=2, label="Decision boundary (0.5)")

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("AI-Generated Score")
ax.set_title("Hive API Scores: Real News (green) vs AI-Generated (red)")
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
plt.show()